# Phishing Detector System Using Random Forest: Training and Evaluation

**Before running:** click **Add Input** (right panel) and add the dataset **"Web page Phishing Detection Dataset"** (Hannousse & Yahiouche, 2021; 11,430 URLs; file `dataset_phishing.csv`, columns `url` and `status`). Then click **Run All**.

The notebook writes `model.joblib`, `results.json`, `confusion_matrix.png` and `feature_importance.png` to the Output tab. Download `model.joblib` and place it beside `app.py`.

In [ ]:
%%writefile features.py
"""
features.py - URL feature extraction and rule-based analysis
Phishing Detector System Using Random Forest (Pepple Christian Boma)

The same functions are used for training (train.py / Kaggle notebook) and for
live detection (app.py), so the model always receives identically computed
features. The URL is analysed as text only; the website is never visited.
"""
import re
from urllib.parse import urlparse

# Order of features fed to the Random Forest model (Table 3.1)
FEATURE_NAMES = [
    "url_length", "num_dots", "num_subdomains", "num_hyphens", "has_at",
    "ip_host", "uses_https", "num_special", "num_digits", "has_keyword",
    "double_slash_path", "is_shortener",
]

SUSPICIOUS_KEYWORDS = [
    "login", "signin", "verify", "secure", "account", "update", "bank",
    "confirm", "password", "webscr", "ebayisapi", "billing", "suspend",
]

SHORTENERS = {
    "bit.ly", "goo.gl", "tinyurl.com", "t.co", "ow.ly", "is.gd", "buff.ly",
    "cutt.ly", "rb.gy", "shorturl.at", "tiny.cc", "rebrand.ly", "s.id",
}

IP_RE = re.compile(r"^(\d{1,3}\.){3}\d{1,3}$")

# Rule-based analyser settings (Table 3.3)
LONG_URL_LIMIT = 75
MAX_SUBDOMAINS = 2
RULE_THRESHOLD = 2


def normalise(url):
    """Strip spaces and add a default scheme so the URL can be parsed."""
    url = (url or "").strip()
    if not re.match(r"^[a-zA-Z][a-zA-Z0-9+.\-]*://", url):
        url = "http://" + url
    return url


def parse(url):
    """Return (normalised_url, parsed, hostname) or raise ValueError."""
    if not url or not url.strip():
        raise ValueError("No URL provided.")
    norm = normalise(url)
    parsed = urlparse(norm)
    host = (parsed.hostname or "").lower()
    if not host:
        raise ValueError("No host name could be found in the input.")
    return norm, parsed, host


def count_subdomains(host):
    if IP_RE.match(host):
        return 0
    labels = [l for l in host.split(".") if l]
    if labels and labels[0] == "www":
        labels = labels[1:]
    # registered domain + top-level domain = last two labels
    return max(len(labels) - 2, 0)


def extract_features(url):
    """Convert a URL into the numerical feature dictionary of Table 3.1."""
    norm, parsed, host = parse(url)
    lower = norm.lower()
    after_scheme = norm.split("://", 1)[1] if "://" in norm else norm
    f = {
        "url_length": len(norm),
        "num_dots": norm.count("."),
        "num_subdomains": count_subdomains(host),
        "num_hyphens": norm.count("-"),
        "has_at": int("@" in norm),
        "ip_host": int(bool(IP_RE.match(host))),
        "uses_https": int(parsed.scheme.lower() == "https"),
        "num_special": sum(norm.count(c) for c in "?=%&~_+!*,;$"),
        "num_digits": sum(ch.isdigit() for ch in norm),
        "has_keyword": int(any(k in lower for k in SUSPICIOUS_KEYWORDS)),
        "double_slash_path": int("//" in after_scheme),
        "is_shortener": int(host in SHORTENERS or host.replace("www.", "") in SHORTENERS),
    }
    return f


def feature_vector(url):
    f = extract_features(url)
    return [f[n] for n in FEATURE_NAMES]


def rule_analyse(url):
    """Apply the heuristic rules of Table 3.3. Returns (score, is_phishing, reasons)."""
    norm, parsed, host = parse(url)
    f = extract_features(url)
    reasons = []
    if f["url_length"] > LONG_URL_LIMIT:
        reasons.append("URL is longer than %d characters" % LONG_URL_LIMIT)
    if f["ip_host"]:
        reasons.append("Host is an IP address")
    if f["has_at"]:
        reasons.append('URL contains "@"')
    if "-" in host:
        reasons.append("Domain name contains a hyphen")
    if f["num_subdomains"] > MAX_SUBDOMAINS:
        reasons.append("More than %d subdomains" % MAX_SUBDOMAINS)
    if f["has_keyword"]:
        reasons.append("Contains a suspicious keyword")
    if not f["uses_https"]:
        reasons.append("Does not use HTTPS")
    if f["is_shortener"]:
        reasons.append("Uses a URL shortening service")
    score = len(reasons)
    return score, score >= RULE_THRESHOLD, reasons


In [ ]:
%%writefile train.py
"""
train.py - trains and evaluates the Random Forest phishing URL classifier.

Usage (local):   python train.py path/to/dataset.csv
On Kaggle the dataset is found automatically under /kaggle/input.

Recommended dataset (2020 or later): "Web page Phishing Detection Dataset"
(Hannousse & Yahiouche, 2021): 11,430 URLs, 50% phishing / 50% legitimate,
columns include 'url' and 'status' (legitimate / phishing).

Outputs: model.joblib, results.json, confusion_matrix.png, feature_importance.png
"""
import glob, json, os, sys, time
import numpy as np
import pandas as pd
import joblib
import matplotlib
matplotlib.use("Agg")
import matplotlib.pyplot as plt
from sklearn.ensemble import RandomForestClassifier
from sklearn.model_selection import train_test_split, StratifiedKFold, cross_val_score
from sklearn.metrics import (accuracy_score, precision_score, recall_score, f1_score,
                             confusion_matrix, roc_auc_score)
from features import FEATURE_NAMES, feature_vector, rule_analyse

RANDOM_STATE = 42
N_TREES = 100
TEST_SIZE = 0.20
OUT = os.environ.get("OUT_DIR", ".")


def find_dataset():
    if len(sys.argv) > 1:
        return sys.argv[1]
    for p in glob.glob("/kaggle/input/**/*.csv", recursive=True):
        return p
    raise SystemExit("No dataset found. Pass the CSV path as an argument.")


def load(path):
    df = pd.read_csv(path)
    cols = {c.lower(): c for c in df.columns}
    url_col = cols.get("url") or cols.get("urls") or cols.get("domain")
    lab_col = next((cols[c] for c in ("status", "label", "type", "result", "class") if c in cols), None)
    if url_col is None or lab_col is None:
        raise SystemExit("Could not find URL / label columns in %s" % list(df.columns))
    df = df[[url_col, lab_col]].rename(columns={url_col: "url", lab_col: "label"}).dropna()

    def to01(v):
        s = str(v).strip().lower()
        if s in ("phishing", "phish", "bad", "malicious", "1"):
            return 1
        if s in ("legitimate", "good", "benign", "0", "safe"):
            return 0
        return None
    df["y"] = df["label"].map(to01)
    df = df.dropna(subset=["y"]).astype({"y": int})
    df["url"] = df["url"].astype(str).str.strip()
    df = df[df["url"] != ""].drop_duplicates(subset="url")
    return df


def metrics(y, p):
    tn, fp, fn, tp = confusion_matrix(y, p, labels=[0, 1]).ravel()
    return {
        "accuracy": round(accuracy_score(y, p) * 100, 2),
        "precision": round(precision_score(y, p, zero_division=0) * 100, 2),
        "recall": round(recall_score(y, p, zero_division=0) * 100, 2),
        "f1": round(f1_score(y, p, zero_division=0) * 100, 2),
        "false_positive_rate": round(fp / (fp + tn) * 100, 2) if (fp + tn) else 0.0,
        "TP": int(tp), "FN": int(fn), "FP": int(fp), "TN": int(tn),
    }


def main():
    path = find_dataset()
    print("Dataset:", path)
    df = load(path)
    print("Usable URLs: %d  (phishing %d, legitimate %d)" % (len(df), df.y.sum(), (df.y == 0).sum()))

    rows, keep = [], []
    for i, u in enumerate(df["url"]):
        try:
            rows.append(feature_vector(u)); keep.append(i)
        except ValueError:
            pass
    df = df.iloc[keep].reset_index(drop=True)
    X = pd.DataFrame(rows, columns=FEATURE_NAMES)
    y = df["y"].values

    X_tr, X_te, y_tr, y_te, u_tr, u_te = train_test_split(
        X, y, df["url"], test_size=TEST_SIZE, stratify=y, random_state=RANDOM_STATE)

    model = RandomForestClassifier(n_estimators=N_TREES, random_state=RANDOM_STATE, n_jobs=-1)
    t0 = time.time(); model.fit(X_tr, y_tr); train_s = time.time() - t0
    t0 = time.time(); p_rf = model.predict(X_te); pred_ms = (time.time() - t0) / len(X_te) * 1000
    prob = model.predict_proba(X_te)[:, 1]

    p_rule = np.array([int(rule_analyse(u)[1]) for u in u_te])
    p_hyb = np.maximum(p_rf, p_rule)          # deployed fusion: phishing if EITHER flags it

    cv = cross_val_score(RandomForestClassifier(n_estimators=N_TREES, random_state=RANDOM_STATE, n_jobs=-1),
                         X, y, cv=StratifiedKFold(5, shuffle=True, random_state=RANDOM_STATE), scoring="accuracy")

    res = {
        "dataset": os.path.basename(path),
        "total_urls": int(len(df)), "phishing": int(y.sum()), "legitimate": int((y == 0).sum()),
        "train_size": int(len(y_tr)), "test_size": int(len(y_te)),
        "n_features": len(FEATURE_NAMES), "n_trees": N_TREES,
        "random_forest": metrics(y_te, p_rf),
        "auc": round(roc_auc_score(y_te, prob), 4),
        "rule_based_only": metrics(y_te, p_rule),
        "hybrid_either": metrics(y_te, p_hyb),
        "cv5_accuracy_mean": round(cv.mean() * 100, 2), "cv5_accuracy_std": round(cv.std() * 100, 2),
        "training_seconds": round(train_s, 2), "prediction_ms_per_url": round(pred_ms, 4),
        "feature_importance": dict(sorted(
            {n: round(float(v), 4) for n, v in zip(FEATURE_NAMES, model.feature_importances_)}.items(),
            key=lambda kv: -kv[1])),
    }

    test_urls = [
        "https://chatgpt.com/codex/get-started/", "http://ux.pu_you-456", "http://ux.pu-you-456",
        "https://www.tripwire.com/state-of-security/evolution-phishing-attacks-why-traditional-detection-methods-are-failing",
        "https://forms.gle/7TZjDVSotiab6LG28",
        "https://www.ijraset.com/research-paper/ai-based-student-performance-prediction-system",
    ]
    res["interface_urls"] = []
    for u in test_urls:
        s, rp, _ = rule_analyse(u)
        mp = int(model.predict(pd.DataFrame([feature_vector(u)], columns=FEATURE_NAMES))[0])
        res["interface_urls"].append({"url": u, "rule_score": s, "rule": int(rp), "model": mp,
                                      "final": int(rp or mp)})

    joblib.dump(model, os.path.join(OUT, "model.joblib"))
    json.dump(res, open(os.path.join(OUT, "results.json"), "w"), indent=2)

    # Figures (black and white)
    cm = np.array([[res["random_forest"]["TP"], res["random_forest"]["FN"]],
                   [res["random_forest"]["FP"], res["random_forest"]["TN"]]])
    fig, ax = plt.subplots(figsize=(4.5, 4))
    ax.imshow(cm, cmap="Greys", alpha=0.35)
    for (i, j), v in np.ndenumerate(cm):
        ax.text(j, i, str(v), ha="center", va="center", fontsize=14)
    ax.set_xticks([0, 1]); ax.set_xticklabels(["Phishing", "Legitimate"]); ax.set_xlabel("Predicted")
    ax.set_yticks([0, 1]); ax.set_yticklabels(["Phishing", "Legitimate"]); ax.set_ylabel("Actual")
    plt.tight_layout(); plt.savefig(os.path.join(OUT, "confusion_matrix.png"), dpi=200); plt.close()

    fi = res["feature_importance"]
    fig, ax = plt.subplots(figsize=(6, 4))
    ax.barh(list(fi.keys())[::-1], list(fi.values())[::-1], color="black")
    ax.set_xlabel("Importance (mean decrease in Gini impurity)")
    plt.tight_layout(); plt.savefig(os.path.join(OUT, "feature_importance.png"), dpi=200); plt.close()

    print(json.dumps(res, indent=2))


if __name__ == "__main__":
    main()


In [ ]:
import glob
print(glob.glob('/kaggle/input/**/*.csv', recursive=True))

In [ ]:
!python train.py $(python -c "import glob;print([p for p in glob.glob('/kaggle/input/**/*.csv',recursive=True) if 'phish' in p.lower()][0])")

In [ ]:
from IPython.display import Image, display
display(Image('confusion_matrix.png'))
display(Image('feature_importance.png'))

In [ ]:
import json, pandas as pd
r=json.load(open('results.json'))
print('Dataset:', r['dataset'], '| URLs:', r['total_urls'], '| phishing:', r['phishing'], '| legitimate:', r['legitimate'])
print('Train/test:', r['train_size'], '/', r['test_size'], '| AUC:', r['auc'], '| 5-fold CV accuracy: %s%% (+/- %s)'%(r['cv5_accuracy_mean'], r['cv5_accuracy_std']))
display(pd.DataFrame({k:r[k] for k in ['random_forest','rule_based_only','hybrid_either']}))
display(pd.DataFrame(r['interface_urls']))
print(json.dumps(r, indent=2))

**Copy the full printed `results.json` from the last cell and send it back** so the real values can be inserted into Chapters Three, Four and Five and the abstract.